In [ ]:
import numpy as np
import os.path as osp
from collections import defaultdict
from sklearn.neighbors import NearestNeighbors  # Students: you can use this implementation to find the 
                                                # Nearest-Neigbors

In [ ]:
# Students: Default location of saved latent codes per last cell of main.ipynb, change appropriately if
# you saved them in another way.
vanilla_ae_emb_file = '../data/out/pc_ae_latent_codes.npz'
part_ae_emb_file = '../data/out/part_pc_ae_latent_codes.npz'

In [ ]:
# Load golden distances (pairwise matrix, or corresponding model/part names in golden_names)
golden_part_dist_file = '../data/golden_dists.npz'
golden_data = np.load(golden_part_dist_file, allow_pickle=True)
golden_part_dist = golden_data['golden_part_dist']
golden_names = golden_data['golden_names']
print(len(golden_names))  # models-name/part combinations
print(golden_names[0])

In [ ]:
# To load vanilla-AE-embeddings (if False will open those of the 2-branch AE).
vanilla = True # or False

In [ ]:
# Load/organize golden part-aware distances.
sn_id_to_parts = defaultdict(list)
id_to_part_loc = dict()

for i, name in enumerate(golden_names):
    # Extract shape-net model ids of golden, map them to their parts.
    sn_id, _, part_id, _, _ = name.split('_')
    sn_id_to_parts[sn_id].append(part_id)
    
    # Map shape-net model id and part_id to location in distance matrix, (the order is the same).
    id_to_part_loc[(sn_id, part_id)] = i

In [ ]:
if vanilla:
    in_d = np.load(vanilla_ae_emb_file)    # Students: assuming you used the numpy.savez
else:
    in_d = np.load(part_ae_emb_file)
        
latent_codes = in_d['latent_codes']
test_names = in_d['test_names']

In [ ]:
# TODO: Use golden distances and matchings to solve question (g)
# Problem (g): compare the part-awareness of the vanilla (d) and part-aware (f)
# encoding spaces. For each test chair A we find its nearest neighbor B in latent
# space (Euclidean) and accumulate the one-way part-based distance D(A, B) from the
# handout, then report the cumulative distance, the average number of shared part
# types, and the average latent NN distance for both embeddings.
import matplotlib.pyplot as plt


def part_based_one_way_distance(a_name, b_name):
    """One-way part-based distance D(A, B) and the number of shared part types."""
    parts_a = set(sn_id_to_parts[a_name])
    parts_b = set(sn_id_to_parts[b_name])
    shared = parts_a & parts_b          # M(A) intersect M(B)
    only_a = parts_a - parts_b          # M(A) \ M(B)

    # Term 1: shared part types are compared like-for-like.
    term1 = 0.0
    for k in shared:
        term1 += golden_part_dist[id_to_part_loc[(a_name, k)],
                                  id_to_part_loc[(b_name, k)]]

    # Term 2: A's parts that B lacks -> assigned to the single B-part u that
    # maximizes the total distance (worst-case term, max outside the sum).
    term2 = 0.0
    if only_a:
        best = -np.inf
        for u in parts_b:
            s = sum(golden_part_dist[id_to_part_loc[(a_name, k)],
                                     id_to_part_loc[(b_name, u)]] for k in only_a)
            best = max(best, s)
        term2 = best

    return term1 + term2, len(shared)


def analyze_embedding(emb_file):
    """Compute the three part-awareness metrics for one saved latent-code file."""
    d = np.load(emb_file, allow_pickle=True)
    codes = d['latent_codes']
    names = [str(x) for x in d['test_names']]

    # Nearest neighbor in latent space (Euclidean). n_neighbors=2 so we can skip self.
    nn = NearestNeighbors(n_neighbors=2, metric='euclidean').fit(codes)
    nn_dists, nn_idxs = nn.kneighbors(codes)

    cumulative, shared_total, latent_dists = 0.0, 0, []
    for i, a_name in enumerate(names):
        b_name = names[nn_idxs[i, 1]]            # nearest *other* chair
        d_ab, n_shared = part_based_one_way_distance(a_name, b_name)
        cumulative += d_ab
        shared_total += n_shared
        latent_dists.append(nn_dists[i, 1])

    n = len(names)
    return {
        'cumulative_part_dist':   cumulative,
        'avg_part_dist_per_chair': cumulative / n,
        'avg_shared_parts':       shared_total / n,
        'avg_latent_nn_dist':     float(np.mean(latent_dists)),
        'n_chairs':               n,
    }


results = {
    'vanilla (d)':    analyze_embedding(vanilla_ae_emb_file),
    'part-aware (f)': analyze_embedding(part_ae_emb_file),
}

print(f"{'metric':<36}{'vanilla (d)':>15}{'part-aware (f)':>17}")
print('-' * 68)
for label, key in [
    ('Cumulative part-based distance',    'cumulative_part_dist'),
    ('Avg part-based distance / chair',   'avg_part_dist_per_chair'),
    ('Avg # shared part types',           'avg_shared_parts'),
    ('Avg latent NN Euclidean distance',  'avg_latent_nn_dist'),
]:
    print(f"{label:<36}{results['vanilla (d)'][key]:>15.4f}{results['part-aware (f)'][key]:>17.4f}")


In [ ]:
# Visual comparison of the three part-awareness metrics (vanilla vs part-aware).
tags   = ['vanilla (d)', 'part-aware (f)']
colors = ['tab:blue', 'tab:orange']
metrics = [
    ('cumulative_part_dist', 'Cumulative part-based distance\n(lower = more part-aware)'),
    ('avg_shared_parts',     'Avg # shared part types\n(higher = more part-aware)'),
    ('avg_latent_nn_dist',   'Avg latent NN Euclidean distance\n(scale differs per embedding)'),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (key, title) in zip(axes, metrics):
    vals = [results[t][key] for t in tags]
    ax.bar(tags, vals, color=colors)
    ax.set_title(title, fontsize=10)
    for i, v in enumerate(vals):
        ax.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../data/out/part_awareness_comparison.png', dpi=150)
plt.show()
